# 1. Import libraries and load the dataset


In [1]:
import sys
from pathlib import Path

# Find the project root containing pyproject.toml
root = Path.cwd()
while root.name and not (root / 'pyproject.toml').exists():
    root = root.parent

if str(root) not in sys.path:
    sys.path.append(str(root))

import pandas as pd
from configs.config import RAW_DIR

df_train = pd.read_csv(RAW_DIR / 'train.csv')


In [2]:
import pandas as pd
import numpy  as np

# 2. Display basic information about the dataset

In [3]:
print("First 5 Rows of Data Frame:\n", df_train.head(5))
print("Data Frame Shape:\n", df_train.shape)
print("Data Frame Info:\n", df_train.info())
print("Data Frame Statistics:\n", df_train.describe())
    

First 5 Rows of Data Frame:
    id        date  store_nbr      family  sales  onpromotion
0   0  2013-01-01          1  AUTOMOTIVE    0.0            0
1   1  2013-01-01          1   BABY CARE    0.0            0
2   2  2013-01-01          1      BEAUTY    0.0            0
3   3  2013-01-01          1   BEVERAGES    0.0            0
4   4  2013-01-01          1       BOOKS    0.0            0
Data Frame Shape:
 (3000888, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB
Data Frame Info:
 None
Data Frame Statistics:
                  id     store_nbr         sales   onpromotion
count  3.000888e+06  3.000888e+06  3.000888e+06  3.000888e+06
mean   1.500444e+0

# 3. Missing Values Summary

In [5]:
missing_counts = df_train.isnull().sum()
missing_percentage = missing_counts / len(df_train) * 100

missing_df_train = pd.DataFrame({
    'Missing Values': missing_counts,
    'Missing Percentage': missing_percentage
})

print("Missing Values Summary:\n", missing_df_train)


missing_df_train = missing_df_train[missing_df_train['Missing Values'] > 0].sort_values(by='Missing Values', ascending=False)

Missing Values Summary:
              Missing Values  Missing Percentage
id                        0                 0.0
date                      0                 0.0
store_nbr                 0                 0.0
family                    0                 0.0
sales                     0                 0.0
onpromotion               0                 0.0


# 4. Detailed Analyze and Statistics

In [7]:
total_sales = df_train['sales'].sum()
avg_sales = df_train['sales'].mean()
median_sales = df_train['sales'].median()
std_sales = df_train['sales'].std()
zero_sales_count = (df_train['sales'] == 0.0).sum()
pct_zero_sales = zero_sales_count / len(df_train) * 100
print(f"Total sales: ${total_sales:,.2f}")
print(f"Average sales: ${avg_sales:.2f}")
print(f"Median sales: ${median_sales:.2f}")
print(f"Std dev sales: ${std_sales:.2f}")
print(f"Zero sales count: {zero_sales_count:,} ({pct_zero_sales:.2f}%)")


# Sales by store and family (sums and means)
sales_by_store = df_train.groupby('store_nbr')['sales'].sum().sort_values(ascending=False)
sales_by_family = df_train.groupby('family')['sales'].sum().sort_values(ascending=False)
sales_by_store_mean = df_train.groupby('store_nbr')['sales'].mean().sort_values(ascending=False)
sales_by_family_mean = df_train.groupby('family')['sales'].mean().sort_values(ascending=False)

print('\nTop 10 stores by sales:\n', sales_by_store.head(10))
print('\nTop 10 families by sales:\n', sales_by_family.head(10))

# Daily sales (time series) - ensure 'date' is datetime
df_train['date'] = pd.to_datetime(df_train['date'])
daily_sales = df_train.groupby('date')['sales'].sum()
print('\nDaily sales sample:\n', daily_sales.head())

# Create a summary DataFrame
sales_summary = pd.DataFrame({
    'total_sales': [total_sales],
    'avg_sales': [avg_sales],
    'median_sales': [median_sales],
    'std_sales': [std_sales],
    'zero_sales_count': [zero_sales_count],
    'pct_zero_sales': [pct_zero_sales]
})
print('\nSales summary:\n', sales_summary)




Total sales: $1,073,644,952.20
Average sales: $357.78
Median sales: $11.00
Std dev sales: $1102.00
Zero sales count: 939,130 (31.30%)

Top 10 stores by sales:
 store_nbr
44    6.208755e+07
45    5.449801e+07
47    5.094831e+07
3     5.048191e+07
49    4.342010e+07
46    4.189606e+07
48    3.593313e+07
51    3.291149e+07
8     3.049429e+07
50    2.865302e+07
Name: sales, dtype: float64

Top 10 families by sales:
 family
GROCERY I        3.434627e+08
BEVERAGES        2.169545e+08
PRODUCE          1.227047e+08
CLEANING         9.752129e+07
DAIRY            6.448771e+07
BREAD/BAKERY     4.213395e+07
POULTRY          3.187600e+07
MEATS            3.108647e+07
PERSONAL CARE    2.459205e+07
DELI             2.411032e+07
Name: sales, dtype: float64

Daily sales sample:
 date
2013-01-01      2511.618999
2013-01-02    496092.417944
2013-01-03    361461.231124
2013-01-04    354459.677093
2013-01-05    477350.121229
Name: sales, dtype: float64

Sales summary:
     total_sales   avg_sales  median_s

# 5. Zero Sales Analysis

Purpose:
- Deep dive into the fact that **31.3% of the observations have zero sales**.
- Determine the percentage of zero-sales transactions by Store (`store_nbr`) and Product Family (`family`).
- Understanding this phenomenon will help guide data processing strategies (e.g., filtering out product families that are never sold at specific stores, or designing a two-stage model that predicts the probability of positive sales before forecasting the exact quantity).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calculate zero sales percentage by Store
zero_sales_by_store = df_train.groupby('store_nbr')['sales'].apply(lambda x: (x == 0).mean() * 100).sort_values(ascending=False)
print("--- Highest Zero Sales Percentage by Store (Top 10 Stores) ---")
print(zero_sales_by_store.head(10))
print("\n--- Lowest Zero Sales Percentage by Store (Top 10 Stores) ---")
print(zero_sales_by_store.tail(10))

# 2. Calculate zero sales percentage by Product Family
zero_sales_by_family = df_train.groupby('family')['sales'].apply(lambda x: (x == 0).mean() * 100).sort_values(ascending=False)
print("\n--- Highest Zero Sales Percentage by Product Family (Top 10 Families) ---")
print(zero_sales_by_family.head(10))
print("\n--- Lowest Zero Sales Percentage by Product Family (Top 10 Families) ---")
print(zero_sales_by_family.tail(10))

# 3. Plot visualizations
fig, axes = plt.subplots(2, 1, figsize=(16, 14))

# Plot Zero Sales by Store
sns.barplot(x=zero_sales_by_store.index, y=zero_sales_by_store.values, ax=axes[0], color='teal', order=zero_sales_by_store.index)
axes[0].set_title('Percentage of Zero Sales by Store', fontsize=14)
axes[0].set_xlabel('Store Number', fontsize=12)
axes[0].set_ylabel('Zero Sales Percentage (%)', fontsize=12)

# Plot Zero Sales by Product Family
sns.barplot(x=zero_sales_by_family.values, y=zero_sales_by_family.index, ax=axes[1], color='coral', order=zero_sales_by_family.index)
axes[1].set_title('Percentage of Zero Sales by Product Family', fontsize=14)
axes[1].set_xlabel('Zero Sales Percentage (%)', fontsize=12)
axes[1].set_ylabel('Product Family', fontsize=12)

plt.tight_layout()
plt.show()

# 6. Missing Dates Analysis

Purpose:
- Check if there are any missing dates globally (e.g., public holidays where stores are closed).
- Check if the time series is regularly spaced and complete for every combination of Store (`store_nbr`) and Product Family (`family`).
- Identifying missing dates is critical for time series forecasting, as model architectures (like SARIMA) assume regularly spaced observations and require imputation or special handling for missing intervals.

In [ ]:
# 1. Find globally missing dates in the dataset
df_train['date'] = pd.to_datetime(df_train['date'])
min_date = df_train['date'].min()
max_date = df_train['date'].max()
expected_dates = pd.date_range(start=min_date, end=max_date)
actual_dates = pd.to_datetime(df_train['date'].unique())

missing_dates = expected_dates.difference(actual_dates)
print(f"Date Range: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
print(f"Expected number of days: {len(expected_dates)}")
print(f"Actual number of unique days in train set: {len(actual_dates)}")
print(f"Number of globally missing days: {len(missing_dates)}")
print("Globally missing dates:")
for d in missing_dates:
    print(f" - {d.strftime('%Y-%m-%d')} (Day of week: {d.day_name()})")

# 2. Verify completeness for each store/family combination
num_stores = df_train['store_nbr'].nunique()
num_families = df_train['family'].nunique()
expected_combinations = num_stores * num_families
actual_combinations = df_train.groupby('date').size().unique()

print(f"\nNumber of stores: {num_stores}")
print(f"Number of product families: {num_families}")
print(f"Expected records per day (Stores x Families): {expected_combinations}")
print(f"Actual records per day found in dataset: {actual_combinations}")

# 3. Check if all store-family time series have the exact same length
series_lengths = df_train.groupby(['store_nbr', 'family']).size()
print(f"\nUnique lengths of time series per (store, family) combination: {series_lengths.unique()}")
if len(series_lengths.unique()) == 1:
    print("SUCCESS: Every (store, family) time series has the exact same length. No locally missing rows!")
else:
    print("WARNING: Some combinations have missing rows. Length distribution:")
    print(series_lengths.value_counts())

# 7. Distribution Analysis (Raw vs. Log Scale)

Purpose: Plot histograms of `sales` on both the raw scale and the log scale (`log1p`) to assess the skewness of the target variable. This will help determine whether the model should be trained on `log1p(sales)` to align directly with the RMSLE competition metric.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set theme and style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Histogram of raw sales (Raw scale)
sns.histplot(df_train['sales'], bins=100, ax=axes[0], kde=True, color='skyblue')
axes[0].set_title('Histogram of Sales (Raw Scale)', fontsize=14)
axes[0].set_xlabel('Sales', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)

# 2. Histogram of log-transformed sales (Log scale)
sns.histplot(np.log1p(df_train['sales']), bins=100, ax=axes[1], kde=True, color='salmon')
axes[1].set_title('Histogram of Sales (Log1p Scale)', fontsize=14)
axes[1].set_xlabel('Log1p(Sales)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)

plt.tight_layout()
plt.show()

# Calculate skewness
skew_raw = df_train['sales'].skew()
skew_log = np.log1p(df_train['sales']).skew()
print(f"Skewness of raw sales: {skew_raw:.2f}")
print(f"Skewness of log1p sales: {skew_log:.2f}")